# TASK #

Laden Sie den UCI Power Consumption-Datensatz (kaggle.com/datasets/uciml/electric-power-consumption-data-set) herunter. Importieren Sie die Zeitreihendaten in PostgreSQL. Fuehren Sie eine explorative Zeitreihenanalyse durch (Saisonalitaet, Trends, Ausreißer). Erstellen Sie mit Python und Facebook Prophet eine Prognose fuer den Energieverbrauch der naechsten 30 Tage. Visualisieren Sie Ist- und Prognosewerte in einem interaktiven Plotly-Dashboard.

## Data from Kaggle ##

In [1]:
import kagglehub
import pandas as pd
import os

from sqlalchemy import create_engine

"""
Lädt den UCI 'Electric Power Consumption' Datensatz via KaggleHub herunter
und liest die CSV-Datei als DataFrame ein. Fehlende Werte ('?') werden als NaN behandelt.
"""

# Download
path = kagglehub.dataset_download("uciml/electric-power-consumption-data-set")
print("Path to dataset files:", path)

# Dateien im Ordner anzeigen
print(os.listdir(path))

# CSV einlesen (Dateiname anpassen falls nötig)
df = pd.read_csv(os.path.join(path, "household_power_consumption.txt"), 
                 sep=";", 
                 na_values="?",
                 low_memory=False)

/Users/luis/Desktop/workdir/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /Users/luis/.cache/kagglehub/datasets/uciml/electric-power-consumption-data-set/versions/1
['household_power_consumption.txt']


### fix timestamp and collumn gramar ###

In [2]:
"""
Bereitet die Rohdaten vor: Kombiniert 'Date' und 'Time' zu einem einzigen Timestamp,
konvertiert 'Date' in ein Datumsobjekt und normalisiert alle Spaltennamen zu Kleinbuchstaben.
"""

df["timestamp"] = pd.to_datetime(
    df["Date"] + " " + df["Time"],
    format="%d/%m/%Y %H:%M:%S"
)

df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)

df.columns = df.columns.str.lower()

## connect to sql and read in data ##

In [ ]:
import os
import psycopg2
from sqlalchemy import create_engine
from sqlalchemy.types import Float, DateTime, Date, Time
from dotenv import load_dotenv

"""
Stellt die Datenbankverbindung via SQLAlchemy her (Credentials aus .env)
und schreibt den DataFrame 'df' in die PostgreSQL-Tabelle 'power_consumption'.
Bestehende Tabelle wird ersetzt. Datentypen werden explizit gesetzt (Float, DateTime, Date, Time).
"""

load_dotenv()

def get_engine():
    """Return SQLAlchemy engine using .env credentials."""
    host = os.getenv("DB_HOST", "localhost")
    port = os.getenv("DB_PORT", "5432")
    user = os.getenv("DB_USER", "luis")
    password = os.getenv("DB_PASSWORD", "")
    dbname = os.getenv("DB_NAME", "power_consumption")
    
    return create_engine(f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}")

# Import
print("Starte Import!")

engine = get_engine()

df.to_sql(
    name="power_consumption",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=10000,
    dtype={
        "date": Date(),
        "time": Time(),
        "timestamp": DateTime(),
        "global_active_power": Float(),
        "global_reactive_power": Float(),
        "voltage": Float(),
        "global_intensity": Float(),
        "sub_metering_1": Float(),
        "sub_metering_2": Float(),
        "sub_metering_3": Float()
    }
)
print("Import fertig!")

**Feature Tabelle**

In [ ]:
from sqlalchemy import text

"""
Erstellt den SQL-View 'power_features' auf Basis der Tabelle 'power_consumption'.
Der View erweitert die Rohdaten um Zeit-Features (Stunde, Wochentag, Monat, Jahr, is_weekend)
sowie berechnete Energie-Spalten (total_sub_metering, unmetered_energy).
"""

engine = get_engine()

with engine.connect() as conn:
    conn.execute(text("DROP VIEW IF EXISTS power_features;"))
    
    conn.execute(text("""
        CREATE VIEW power_features AS
        SELECT
            timestamp,
            global_active_power,
            global_reactive_power,
            voltage,
            global_intensity,
            sub_metering_1,
            sub_metering_2,
            sub_metering_3,
            EXTRACT(HOUR FROM timestamp) AS hour,
            EXTRACT(DOW FROM timestamp) AS weekday,
            EXTRACT(MONTH FROM timestamp) AS month,
            EXTRACT(YEAR FROM timestamp) AS year,
            CASE
                WHEN EXTRACT(DOW FROM timestamp) IN (0,6)
                THEN 1
                ELSE 0
            END AS is_weekend,
            sub_metering_1 + sub_metering_2 + sub_metering_3 AS total_sub_metering,
            (global_active_power * 1000 / 60)
            - (sub_metering_1 + sub_metering_2 + sub_metering_3)
            AS unmetered_energy
        FROM power_consumption;
    """))
    conn.commit()

print("View erstellt!")

View erstellt!


**Stundenaggregation**

In [ ]:
"""
Erstellt den SQL-View 'hourly_consumption'.
Aggregiert die Minutendaten aus 'power_features' auf Stundenbasis –
mit Durchschnitts- und Spitzenleistung sowie den drei Sub-Meterings und ungemessenem Verbrauch.
"""

with engine.connect() as conn:
    conn.execute(text("DROP VIEW IF EXISTS hourly_consumption;"))
    conn.execute(text("""
        CREATE VIEW hourly_consumption AS
        SELECT
            DATE_TRUNC('hour', timestamp) AS hour,
            AVG(global_active_power) AS avg_power,
            MAX(global_active_power) AS peak_power,
            AVG(sub_metering_1) AS sub1,
            AVG(sub_metering_2) AS sub2,
            AVG(sub_metering_3) AS sub3,
            AVG(unmetered_energy) AS unmetered_energy
        FROM power_features
        GROUP BY DATE_TRUNC('hour', timestamp)
        ORDER BY DATE_TRUNC('hour', timestamp);
    """))
    conn.commit()

print("View erstellt!")

View erstellt!


**Tagesaggregation**

In [ ]:
"""
Erstellt den SQL-View 'daily_consumption'.
Aggregiert die Daten aus 'power_features' auf Tagesbasis –
mit Durchschnitts- und Gesamtleistung sowie den drei Sub-Meterings und ungemessenem Verbrauch.
"""

with engine.connect() as conn:
    conn.execute(text("DROP VIEW IF EXISTS daily_consumption;"))
    conn.execute(text("""
        CREATE VIEW daily_consumption AS
        SELECT
            DATE(timestamp) AS day,
            AVG(global_active_power) AS avg_power,
            SUM(global_active_power) AS total_power,
            AVG(sub_metering_1) AS sub1,
            AVG(sub_metering_2) AS sub2,
            AVG(sub_metering_3) AS sub3,
            AVG(unmetered_energy) AS unmetered_energy
        FROM power_features
        GROUP BY DATE(timestamp)
        ORDER BY DATE(timestamp);
    """))
    conn.commit()

print("View erstellt!")

View erstellt!


**Verbrauch nach Stunden des Tages**

In [ ]:
"""
Erstellt den SQL-View 'hourly_pattern'.
Berechnet den durchschnittlichen Stromverbrauch pro Stunde des Tages (0–23)
aus 'power_features' – zeigt typische Tagesverbrauchsmuster.
"""

with engine.connect() as conn:
    conn.execute(text("DROP VIEW IF EXISTS hourly_pattern;"))
    conn.execute(text("""
        CREATE VIEW hourly_pattern AS
        SELECT
            hour,
            AVG(global_active_power) AS avg_power
        FROM power_features
        GROUP BY hour
        ORDER BY hour;
    """))
    conn.commit()

print("View erstellt!")

View erstellt!


**Verbrauch nach Wochentagen**

In [ ]:
"""
Erstellt den SQL-View 'weekday_pattern'.
Berechnet den durchschnittlichen Stromverbrauch pro Wochentag (0=Sonntag, 6=Samstag)
aus 'power_features' – zeigt typische Wochenverbrauchsmuster.
"""

with engine.connect() as conn:
    conn.execute(text("DROP VIEW IF EXISTS weekday_pattern;"))
    conn.execute(text("""
        CREATE VIEW weekday_pattern AS
        SELECT
            weekday,
            AVG(global_active_power) AS avg_power
        FROM power_features
        GROUP BY weekday
        ORDER BY weekday;
    """))
    conn.commit()

print("View erstellt!")

View erstellt!


**Monatliche Saisonalität**

In [ ]:
"""
Erstellt den SQL-View 'monthly_pattern'.
Berechnet den durchschnittlichen Stromverbrauch pro Monat (1=Januar, 12=Dezember)
aus 'power_features' – zeigt saisonale Verbrauchsmuster über das Jahr.
"""

with engine.connect() as conn:
    conn.execute(text("DROP VIEW IF EXISTS monthly_pattern;"))
    conn.execute(text("""
        CREATE VIEW monthly_pattern AS
        SELECT
            month,
            AVG(global_active_power) AS avg_power
        FROM power_features
        GROUP BY month
        ORDER BY month;
    """))
    conn.commit()

print("View erstellt!")

View erstellt!


**Submetering Struktur**

In [ ]:
"""
Erstellt den SQL-View 'submeter_distribution'.
Berechnet den durchschnittlichen Verbrauch je Bereich (Küche, Waschküche, Warmwasser/Klima, Sonstiges)
aus 'power_features' – gibt einen Überblick über die Verbrauchsverteilung im Haushalt.
"""

with engine.connect() as conn:
    conn.execute(text("DROP VIEW IF EXISTS submeter_distribution;"))
    conn.execute(text("""
        CREATE VIEW submeter_distribution AS
        SELECT
            AVG(sub_metering_1) AS kitchen_usage,
            AVG(sub_metering_2) AS laundry_usage,
            AVG(sub_metering_3) AS water_heater_ac,
            AVG(unmetered_energy) AS other_usage
        FROM power_features;
    """))
    conn.commit()

print("View erstellt!")

View erstellt!


**Peak Verbrauchstage**

In [ ]:
"""
Erstellt den SQL-View 'peak_days'.
Ermittelt die 20 Tage mit dem höchsten Spitzenverbrauch (MAX global_active_power)
aus 'power_features' – sortiert absteigend nach Spitzenleistung.
"""

with engine.connect() as conn:
    conn.execute(text("DROP VIEW IF EXISTS peak_days;"))
    conn.execute(text("""
        CREATE VIEW peak_days AS
        SELECT
            DATE(timestamp) AS day,
            MAX(global_active_power) AS peak_power
        FROM power_features
        GROUP BY DATE(timestamp)
        ORDER BY peak_power DESC
        LIMIT 20;
    """))
    conn.commit()

print("View erstellt!")

View erstellt!


**Hidden Energy Analyse**

In [ ]:
"""
Erstellt den SQL-View 'hidden_consumption'.
Berechnet den durchschnittlichen ungemessenen Verbrauch (unmetered_energy) pro Tag
aus 'power_features' – zeigt den täglich nicht zugeordneten Stromanteil im Zeitverlauf.
"""

with engine.connect() as conn:
    conn.execute(text("DROP VIEW IF EXISTS hidden_consumption;"))
    conn.execute(text("""
        CREATE VIEW hidden_consumption AS
        SELECT
            DATE(timestamp) AS day,
            AVG(unmetered_energy) AS avg_hidden_energy
        FROM power_features
        GROUP BY DATE(timestamp)
        ORDER BY DATE(timestamp);
    """))
    conn.commit()

print("View erstellt!")

View erstellt!
